<a href="https://colab.research.google.com/github/Ademola-Olorunnisola/TB-Estimator/blob/main/observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.integrate import odeint
import scipy.optimize

import sympy as sp

import pandas as pd

import copy

In [5]:

try:
    import casadi
except:
    !pip install casadi
    import casadi

try:
    import do_mpc
except:
    !pip install do_mpc
    import do_mpc

try:
    import pybounds
except:
    #!pip install pybounds
    !pip install git+https://github.com/vanbreugel-lab/pybounds
    import pybounds

### Import plotting utilities and planar drone locally or from github

In [6]:
import sys
import requests
import importlib

def import_local_or_github(package_name, function_name=None, directory=None, giturl=None):
    # Import functions directly from github
    # Important: note that we use raw.githubusercontent.com, not github.com

    try: # to find the file locally
        if directory is not None:
            if directory not in sys.path:
                sys.path.append(directory)

        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package, function_name)
            return function
        else:
            return package

    except: # get the file from github
        if giturl is None:
            giturl = 'https://raw.githubusercontent.com/Ademola-Olorunnisola/TB-Estimator/main/Utility/' + str(package_name) + '.py'

        r = requests.get(giturl)
        print('Fetching from: ')
        print(r)

        # Store the file to the colab working directory
        with open(package_name+'.py', 'w') as f:
            f.write(r.text)
        f.close()

        # import the function we want from that file
        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package , function_name)
            return function
        else:
            return package

tb_simulations = import_local_or_github('model_simulation', directory='../Utility')
plot_tme = import_local_or_github('plot_utility', 'plot_tme', directory='../Utility')

Fetching from: 
<Response [200]>


NameError: name 'N' is not defined

# TB DYNAMICS


$$
\mathbf{\dot{x}} = \mathbf{f}(\mathbf{x},\mathbf{u}) = \mathbf{f_0}(\mathbf{x}) + \mathbf{f_1}(\mathbf{x})\bbox[lightgreen]{\alpha} + \mathbf{f_2}(\mathbf{x})\bbox[lightgreen]{\kappa}
$$

$$
\frac{d}{dt}
\begin{bmatrix}
\bbox[yellow]{S} \\[0.3em]
\bbox[yellow]{V} \\[0.3em]
\bbox[yellow]{I} \\[0.3em]
\bbox[yellow]{R} \\[0.3em]
\bbox[pink]{\beta} \\[0.3em]
\bbox[pink]{\sigma}
\end{bmatrix} =
\overset{f_0}{\begin{bmatrix}
\bbox[lightblue]{\Lambda} - \bbox[pink]{\beta}\bbox[yellow]{SI} - \bbox[lightblue]{\mu}\bbox[yellow]{S} \\[0.3em]
0 - \bbox[pink]{\sigma}\bbox[pink]{\beta}\bbox[yellow]{VI} - \bbox[lightblue]{\mu}\bbox[yellow]{V} \\[0.3em]
\bbox[pink]{\beta}\bbox[yellow]{SI} + \bbox[pink]{\sigma}\bbox[pink]{\beta}\bbox[yellow]{VI} - \bbox[lightblue]{\gamma}\bbox[yellow]{I} - \bbox[lightblue]{\mu}\bbox[yellow]{I} \\[0.3em]
\bbox[lightblue]{\gamma}\bbox[yellow]{I} - \bbox[lightblue]{\mu}\bbox[yellow]{R}\\[0.3em]
0\\[0.3em]
0
\end{bmatrix}} +
\overset{f_1}{\begin{bmatrix}
-\bbox[yellow]{S} \\[0.3em]
\bbox[yellow]{S} \\[0.3em]
0 \\[0.3em]
0\\[0.3em]
0\\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{\alpha} +
\overset{f_2}{\begin{bmatrix}
\bbox[pink]{\beta}\bbox[yellow]{SI} \\[0.3em]
\bbox[pink]{\sigma}\bbox[pink]{\beta}\bbox[yellow]{VI} \\[0.3em]
-\bbox[pink]{\beta}\bbox[yellow]{SI} - \bbox[pink]{\sigma}\bbox[pink]{\beta}\bbox[yellow]{VI} \\[0.3em]
0\\[0.3em]
0\\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{\kappa}
$$

# Dynamics and measurement functions

In [ ]:
h_a = tb_simulations.H('observe_IVR').h
h_b = tb_simulations.H('observe_IVRЕТ').h
h_c = tb_simulations.H('observe_I').h


In [ ]:
f = tb_simulations.F().f
h = h_b

In [ ]:
def u_func(x_vec, t):
    """Vaccination and treatment rate — matched to UKF implementation"""
    alpha = 0.01    # u1: BCG vaccination rate
    tau   = 0.005     # u2: Treatment initiation rate (R₀ = 0.87, mild control)
    return np.array([alpha, tau])

In [ ]:
def f_ode(x_vec, t):
    """Wrapper for odeint"""
    u_vec = u_func(x_vec, t)
    return f(x_vec, u_vec)

In [ ]:
N_pop = 220e6

# True parameter values
beta_true  = 0.15
sigma_true = 0.40
gamma_true = 1 / 180
eps_true   = 1 / 365
phi_true   = 0.03
rho_true   = 0.21 * gamma_true

# Initial compartments (normalised proportions × N for odeint)
I0    = 361000 / N_pop
V0    = 0.65
R0_ic = 0.02
E0    = 3 * I0
T0    = 0.6 * I0
S0    = 1.0 - V0 - E0 - I0 - T0 - R0_ic

# Augmented state: [S, V, E, I, T, R, beta, sigma, gamma, eps, phi, rho]
x0 = np.array([
    S0,        V0,         E0,       I0,
    T0,        R0_ic,
    beta_true, sigma_true, gamma_true, eps_true,
    phi_true,  rho_true,
])


# Run simulation

In [ ]:
t_sim = np.arange(0, 365, 1.0)
result = odeint(f_ode, x0, t_sim)


In [ ]:
x_sim = {
    'S':     result[:, 0],
    'V':     result[:, 1],
    'E':     result[:, 2],
    'I':     result[:, 3],
    'T':     result[:, 4],
    'R':     result[:, 5],
    'beta':  result[:, 6],
    'eps':   result[:, 7],
    'gamma': result[:, 8],
    'sigma': result[:, 9],
    'phi':   result[:, 10],
    'rho':   result[:, 11],
}

In [ ]:
u_sim = {'alpha': [], 'tau': []}
for i in range(len(t_sim)):
    u = u_func(result[i], t_sim[i])
    u_sim['alpha'].append(u[0])
    u_sim['tau'].append(u[1])

In [ ]:
y_sim_absolute = {
    'S_absolute': x_sim['S'],
    'V_absolute': x_sim['V'],
    'E_absolute': x_sim['E'],
    'I_absolute': x_sim['I'],
    'T_absolute': x_sim['T'],
    'R_absolute': x_sim['R'],
    'beta': x_sim['beta'],
    'eps': x_sim['eps'],
    'gamma': x_sim['gamma'],
    'sigma': x_sim['sigma'],
    'phi': x_sim['phi'],
    'rho': x_sim['rho']
}

### Define measurement noise characteritics for each measurement option in the measurement set

In [ ]:
measurement_noise_stds = {
    'S_absolute': 1000.0,
    'I_absolute': 2500.0,
    'V_absolute': 4000.0,
    'R_absolute': 500.0,
    'E_absolute': 4000.0,
    'T_absolute': 3000.0,
    'beta':  1000,
    'eps':   3000,
    'gamma': 10000,
    'sigma': 4500,
    'phi':   2200,
    'rho':   10000,
}

In [ ]:
# Add noise
np.random.seed(42)
y_noisy_absolute = {
    key: y_sim_absolute[key] + np.random.normal(0, measurement_noise_stds[key], len(t_sim))
    for key in y_sim_absolute.keys()
}

In [ ]:
y_noisy = {key: y_noisy_absolute[key]  for key in y_noisy_absolute.keys()}

### Plot the infected population over time

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(t_sim, x_sim['R'])
ax.set_xlabel('Time (days)')
ax.set_ylabel('Infected Population')
plt.show()

### Save data as dataframes

In [ ]:
x_sim_df = pd.DataFrame(x_sim)
y_noisy_df = pd.DataFrame(y_noisy)
u_sim_df = pd.DataFrame(u_sim)

# Pybound Simulator

In [ ]:
state_names = f(None, None, return_state_names=True)
input_names = ['alpha', 'kappa']
measurement_names = h(None, None, return_measurement_names=True)

In [ ]:
simulator = pybounds.Simulator(
    f,
    h,
    dt=1.0,
    state_names=state_names,
    input_names=input_names,
    measurement_names=measurement_names,
    mpc_horizon=10
)

#Observability analysis

In [ ]:
w = 20  # window size, set to None to use entire time-series as one window

### This is the computationally heavy step

Here we calculate the observability matrix for each sliding window, all the states, and measurements. Later we can subselect from this to choose specific time steps, measurements, states, etc. to consider.

In [ ]:
# Construct O in sliding windows
SEOM = pybounds.SlidingEmpiricalObservabilityMatrix(simulator, t_sim, x_sim, u_sim, w=w, eps=1e-4)

# Get O's
O_sliding = SEOM.get_observability_matrix()

In [ ]:
n_window = len(O_sliding)
print(n_window, 'windows')

### Look at one of the observability matrices

In [ ]:
O_sliding[0]

### Convert a single observability matrix into fisher information

$F = \mathcal{O}_w^T R_w^{-1} \mathcal{O}_w$

In [ ]:
measurement_noise_vars = {key: val**2 for key, val in measurement_noise_stds.items()}

In [ ]:
# CORRECTED: Define measurement noise with std=1 (professor's recommendation)
# This automatically matches whatever measurements your h function returns
measurement_names = h(None, None, return_measurement_names=True)
measurement_noise_stds = {name: 1.0 for name in measurement_names}

# Convert to variances
measurement_noise_vars = {key: val**2 for key, val in measurement_noise_stds.items()}

# Now compute Fisher information
FO = pybounds.FisherObservability(SEOM.O_df_sliding[0], measurement_noise_vars, lam=1e-8)

In [ ]:
# Get the Fisher information, inverse, and R matrix
F, F_inv, R = FO.get_fisher_information()
F_inv

### Efficiently repeat the above process for a specific set of sensors, states, and time points

In [ ]:

# Choose sensors to use from O -- you can select a subset from the available measurements
o_sensors = h(None, None, return_measurement_names=True)  # ['I_absolute'] or ['I_absolute', 'R_absolute']

# Choose states to use from O
o_states = ['S', 'V', 'E', 'I', 'R', 'beta', 'eps', 'gamma', 'sigma', 'phi', 'rho']

# Choose time-steps to use from O
window_size = 7  # this cannot be larger than what was defined above
o_time_steps = np.arange(0, window_size, step=1)

# Redefine R -- if you remove a sensor you need to change R
o_measurement_noise_vars = {key: measurement_noise_vars[key] for key in o_sensors}

In [ ]:
# Compute the Fisher information & inverse for each window and store the minimum error variance
SFO = pybounds.SlidingFisherObservability(SEOM.O_df_sliding, time=SEOM.t_sim, lam=1e-8, R=o_measurement_noise_vars,
                                          states=o_states, sensors=o_sensors, time_steps=o_time_steps, w=None)

# If you want to manually inspect one of the fisher info matrices:
# SFO.FO[1].O

In [ ]:
# Pull out minimum error variance, 'time' column is the time vector shifted forward by w/2 and 'time_initial' is the original time
EV_aligned = SFO.get_minimum_error_variance()

In [ ]:
EV_no_nan = EV_aligned.fillna(method='bfill').fillna(method='ffill')

# Plot observability over time

In [ ]:
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import numpy as np

states = list(SFO.FO[0].O.columns)
n_state = len(states)
fig, ax = plt.subplots(n_state, 2, figsize=(6, n_state*2), dpi=150, sharex=True)
ax = np.atleast_2d(ax)

cmap = 'inferno_r'
min_ev = np.min(EV_no_nan.iloc[:, 2:].values)
max_ev = np.max(EV_no_nan.iloc[:, 2:].values)
log_tick_high = int(np.ceil(np.log10(max_ev)))
log_tick_low = int(np.floor(np.log10(min_ev)))
cnorm = mpl.colors.LogNorm(10**log_tick_low, 10**log_tick_high)

# Formatter for scientific notation
def fmt(x, pos):
    return f'{x:.2e}'

for n, state_name in enumerate(states):
    # Left panel: state trajectory
    pybounds.colorline(t_sim, x_sim[state_name], EV_no_nan[state_name].values,
                       ax=ax[n, 0], cmap=cmap, norm=cnorm)

    # Right panel: eigenvalues
    pybounds.colorline(t_sim, EV_no_nan[state_name].values, EV_no_nan[state_name].values,
                       ax=ax[n, 1], cmap=cmap, norm=cnorm)

    # Colorbar
    cax = ax[n, -1].inset_axes([1.03, 0.0, 0.04, 1.0])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax,
                        ticks=np.logspace(log_tick_low, log_tick_high,
                                         log_tick_high-log_tick_low + 1))
    cbar.set_label('min. EV: ' + state_name, rotation=270, fontsize=7, labelpad=8)
    cbar.ax.tick_params(labelsize=6)

    # IMPROVED: Better y-axis limits for left panel
    x_vals = x_sim[state_name]
    x_range = np.max(x_vals) - np.min(x_vals)

    if x_range < 1e-4:  # Very small changes (like I, R, beta, sigma)
        # Use tighter limits to show variation
        x_mean = np.mean(x_vals)
        padding = max(x_range * 0.5, 1e-5)  # At least some padding
        ax[n, 0].set_ylim(x_mean - padding, x_mean + padding)
    else:  # Larger changes (like S, V)
        x_max = np.max(x_vals)
        x_min = np.min(x_vals)
        padding = x_range * 0.1
        ax[n, 0].set_ylim(x_min - padding, x_max + padding)

    ax[n, 0].set_ylabel('state: ' + state_name, fontsize=7)
    ax[n, 0].yaxis.set_major_formatter(FuncFormatter(fmt))
    ax[n, 0].grid(True, alpha=0.3)

    # Right panel formatting
    ax[n, 1].set_ylim(10**log_tick_low, 10**log_tick_high)
    ax[n, 1].set_yscale('log')
    ax[n, 1].set_ylabel('min. EV: ' + state_name, fontsize=7)
    ax[n, 1].set_yticks(np.logspace(log_tick_low, log_tick_high,
                                    log_tick_high-log_tick_low + 1))
    ax[n, 1].grid(True, alpha=0.3, which='both')

# Final formatting
for a in ax.flat:
    a.tick_params(axis='both', labelsize=6)
    a.set_xlabel('time (days)', fontsize=7)
    offset = t_sim[-1] * 0.05
    a.set_xlim(-offset, t_sim[-1] + offset)

fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.3, hspace=0.4)
plt.show()

# Exercises:

1. Change the measurements to one of the other options listed at the top. What changes for the observability?
2. Change the noise characteristics, how does this change the observability?